# Titanic Mini EDA: Task 3
### Data Science / Data Analysis with Python Internship, Main Crafts Technology

**Workflow:** Load → Clean → Analyze → Visualize → Conclude

In Task 2 we looked at survival by gender and class. Here we go deeper with a **Mini Exploratory Data Analysis (EDA)**: better missing-value handling, `groupby` insights by **age group**, **embarkation port** and **family size**, and more advanced visuals (histogram, correlation heatmap, family-size bar plot).

Dataset: Kaggle Titanic `train.csv` (891 passengers).

## 1. Load Dataset

`train.csv` is downloaded from the [Kaggle Titanic competition page](https://www.kaggle.com/c/titanic/data). If the file is not already in the working folder, Colab will prompt you to upload it.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
os.makedirs('visuals', exist_ok=True)

# train.csv downloaded from Kaggle: https://www.kaggle.com/c/titanic/data
DATA_PATH = 'train.csv'
if not os.path.exists(DATA_PATH):
    from google.colab import files
    files.upload()  # select train.csv when prompted

df = pd.read_csv(DATA_PATH)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
print("Shape of dataset:", df.shape)
df.info()

Shape of dataset: (891, 12)
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


## 2. Clean Dataset

Real-world data is rarely complete. We first measure what is missing, then decide how to treat each column.

In [3]:
# Missing values before cleaning
print("Missing values before cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Missing values before cleaning:
Age         177
Cabin       687
Embarked      2
dtype: int64


**Observation:** `Age` is missing for 177 passengers, `Cabin` for 687 (about 77% of rows) and `Embarked` for 2. Our plan:
- **Age** → fill with the **mean** age, so we keep all 891 passengers.
- **Cabin** → drop; too sparse to be useful.
- **Ticket** → drop; ticket codes carry no analytical value here.
- **Embarked** → fill the 2 gaps with the most common port (mode).

In [4]:
# Fill missing Age with the mean
df['Age'] = df['Age'].fillna(df['Age'].mean())

# Fill missing Embarked with the most common port
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Drop irrelevant columns
df = df.drop(columns=['Cabin', 'Ticket'])

print("Missing values after cleaning:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Mean age used for imputation:", round(df['Age'].mean(), 2))
df.head()

Missing values after cleaning: 0
Duplicate rows: 0
Mean age used for imputation: 29.7


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,8.0500,S


**Observation:** The dataset is now complete: no missing values and no duplicates. One caveat: filling 177 ages with the mean (about 29.7) pulls many passengers to the same value, which slightly narrows the age distribution. We keep this in mind when reading the age charts.